# Capitolo 5 — Un dataset industriale: difetti su acciaio laminato (§ 5.10)
Prima eseguire `python scarica_difetti.py`. Stesso codice del transfer learning gatti/cani, sei classi.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

import time, json
from torchvision import datasets, models
from torch.utils.data import DataLoader
fissa_seme(42)
pesi = models.ResNet18_Weights.DEFAULT; prep = pesi.transforms()
ds = {s: datasets.ImageFolder(f"difetti/{s}", transform=prep) for s in ("train", "val", "test")}
dl = {s: DataLoader(d, batch_size=32, shuffle=False) for s, d in ds.items()}
classi = ds["train"].classes; print(classi, {s: len(d) for s, d in ds.items()})

In [ ]:
from PIL import Image
import os
fig, ax = plt.subplots(2, 6, figsize=(12, 4))
for j, c in enumerate(classi):
    for r in range(2):
        f = sorted(os.listdir(f"difetti/train/{c}"))[r]; ax[r, j].imshow(Image.open(f"difetti/train/{c}/{f}").convert("L"), cmap="gray"); ax[r, j].axis("off")
        if r == 0: ax[r, j].set_title(c)
plt.show()

## Caratteristiche estratte una volta sola dalla ResNet18 congelata

In [ ]:
modello = models.resnet18(weights=pesi); modello.fc = nn.Identity(); modello.eval()
caratt = {}; t0 = time.time()
with torch.no_grad():
    for s in ("train", "val", "test"):
        xs, ys = [], []
        for xb, yb in dl[s]: xs.append(modello(xb)); ys.append(yb)
        caratt[s] = (torch.cat(xs), torch.cat(ys)); print(s, caratt[s][0].shape, f"{time.time() - t0:.0f} s")

In [ ]:
fissa_seme(42)
testa = nn.Linear(512, len(classi)); perdita_fn = nn.CrossEntropyLoss(); opt = torch.optim.Adam(testa.parameters(), lr=1e-3)
X_tr, y_tr = caratt["train"]; X_va, y_va = caratt["val"]; X_te, y_te = caratt["test"]
def accuratezza(X, y):
    with torch.no_grad(): return (testa(X).argmax(1) == y).float().mean().item()
migliore = (0, None, 0)
for epoca in range(60):
    perm = torch.randperm(len(X_tr))
    for i in range(0, len(X_tr), 32):
        idx = perm[i:i + 32]; opt.zero_grad(); perdita_fn(testa(X_tr[idx]), y_tr[idx]).backward(); opt.step()
    av = accuratezza(X_va, y_va)
    if av > migliore[0]: migliore = (av, {k: v.clone() for k, v in testa.state_dict().items()}, epoca + 1)
testa.load_state_dict(migliore[1])
print(f"migliore: epoca {migliore[2]} (val {migliore[0]:.1%})   test: {accuratezza(X_te, y_te):.1%}")

In [ ]:
from sklearn.metrics import confusion_matrix
with torch.no_grad(): pred = testa(X_te).argmax(1).numpy()
C = confusion_matrix(y_te.numpy(), pred)
fig, ax = plt.subplots(figsize=(5, 4.5)); ax.imshow(C, cmap="Blues")
for i in range(6):
    for j in range(6): ax.text(j, i, C[i, j], ha="center", va="center", color="white" if C[i, j] > 30 else "black")
ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_xticklabels(classi, rotation=45, ha="right"); ax.set_yticklabels(classi); ax.set_xlabel("prevista"); ax.set_ylabel("vera"); plt.show()
modello.fc = testa; torch.save(modello.state_dict(), "difetti_resnet18.pt"); json.dump({"classi": classi}, open("difetti.json", "w"))